# APICE Tutorial Notebook

This notebook demonstrates APICE EEG preprocessing and segmentation workflows.

It has two goals:
1. Show high-level pipeline wrappers for multiple files.
2. Show lower-level RawAPICE and EpochsAPICE step-by-step methods.

The examples below are designed for this repository layout and use test data under `test_data/` when available.

## 1) Install and Import Dependencies

If your environment already contains these packages, you can skip the install cell.

This section imports pipeline wrappers (`run_preprocessing`, `run_segmentation`, `preprocess_initial_steps`, `preprocess_apice_default`) and low-level APICE APIs used later.

In [ ]:
# Optional: install dependencies in the active environment
# %pip install mne mne-bids tabulate

from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mne

from apice.pipeline import (
    preprocess_initial_steps,
    preprocess_apice_default,
    segment_default_pipeline,
)
from apice.io import load_rawapice, load_epochapice
from apice.data_structures import RawAPICE, EpochsAPICE
from apice.filter import Filter
from apice.artifacts_rejection import ArtifactsConfiguration, run_algorithms

print("Imports OK")

## 2) Configure Input and Output Directories

Set paths for input and outputs. By default, this notebook points to repository test data.

In [ ]:
# Assume notebook runs from repository root
repo_root = Path.cwd()

# Default local test-data paths
input_dir = repo_root / "test_data" / "raw"
output_dir_preproc = repo_root / "tests" / "preprocessed"
output_dir_segmented = repo_root / "tests" / "segmented"

output_dir_preproc.mkdir(parents=True, exist_ok=True)
output_dir_segmented.mkdir(parents=True, exist_ok=True)

print("input_dir:", input_dir)
print("output_dir_preproc:", output_dir_preproc)
print("output_dir_segmented:", output_dir_segmented)
print("raw test file exists:", (input_dir / "test_recording-raw.fif").exists())

## 3) Set Preprocessing Parameters

Define shared preprocessing parameters used by wrapper and low-level examples.

In [ ]:
# Parameters matched to the test_recording.fif data
# (The FIF file already contains an embedded montage, so montage=None.)

drop_electrodes = ['E125', 'E126', 'E127', 'E128']   # specify channels to drop a priori (e.g., outer ring channels)
reference_channels = ['VREF']                        # test recording  includes the reference channel; indicate it so it is not treated as bad channel
picks = "eeg"
crop_times = None
crop_from_beginnning = None
crop_from_end = None
resample_freq = None
stim_channels_to_annotations = False   # test recording does not use STIMs for events

# montage=None because test_recording.fif already has the montage stored inside.
# For external files (e.g. .vhdr) without an embedded montage, provide a path:
#   montage = Path(repo_root) / "electrode_layout" / "GSN-HydroCel-129.sfp")
montage = None

l_freq = 0.10
h_freq = 40

print("drop_electrodes:", drop_electrodes)
print("reference_channels:", reference_channels)
print("montage:", montage)

## 4) Configure Artifact Detection

Use `None` to load APICE packaged defaults from `apice/default_cfg`.

You can also pass dictionary configs or JSON paths.

In [ ]:
cfg_bad_channels_detection = None
cfg_glitches_detection = None
cfg_target_pca = None
cfg_artifacts_detection = None
cfg_spline_segments = None
cfg_spline_channels = None

# Segmentation-side default cfgs
cfg_define_bcbt_epochs = None
cfg_bad_epochs = None

print("All cfg_* are set to None (APICE defaults).")

## 5) Run Preprocessing Pipeline

This section demonstrates both:
- single-file wrappers (`preprocess_initial_steps`, `preprocess_apice_default`)
- batch wrapper (`run_preprocessing`) for multiple files.

`data_selection_method='all'` processes all eligible input files.
`data_selection_method='new'` processes only files not already present in output.

In [ ]:
# Single-file wrapper demonstration
single_raw_path = input_dir / "test_recording-raw.fif"

if single_raw_path.exists():
    raw_single = mne.io.read_raw(single_raw_path, preload=False, verbose=False)

    raw_single, report_initial = preprocess_initial_steps(
        raw_single,
        output_dir=output_dir_preproc,
        file_name=single_raw_path.stem,
        drop_electrodes=drop_electrodes,
        picks=picks,
        crop_times=crop_times,
        crop_from_beginnning=crop_from_beginnning,
        crop_from_end=crop_from_end,
        resample_freq=resample_freq,
        stim_channels_to_annotations=stim_channels_to_annotations,
        montage=montage,
        create_report=True,
        save_log=False,
        save_cfg=True,
        save_data=False,
        save_report=True,
    )

    raw_apice_single, summary_single, report_single = preprocess_apice_default(
        raw_single,
        output_dir=output_dir_preproc,
        file_name=single_raw_path.stem,
        create_report=True,
        save_log=False,
        save_data=True,
        save_report=True,
        save_summary=True,
        save_cfg=True,
        reference_channels=reference_channels,
        l_freq=l_freq,
        h_freq=h_freq,
        cfg_bad_channels_detection=cfg_bad_channels_detection,
        cfg_glitches_detection=cfg_glitches_detection,
        cfg_target_pca=cfg_target_pca,
        cfg_artifacts_detection=cfg_artifacts_detection,
        cfg_spline_segments=cfg_spline_segments,
        cfg_spline_channels=cfg_spline_channels,
        n_jobs=-1,
    )
    print("Single-file preprocessing finished.")
else:
    print(f"Raw test file not found: {single_raw_path}")

## 6) Run Segmentation Pipeline

Define event extraction settings and run segmentation wrappers over preprocessed files.

In [ ]:
l_freq_epochs = 0.2
h_freq_epochs = 20

kwargs_events_from_annotations_for_segmentation = {
    "regexp": 'animal*',  
    "verbose": False,
}
event_time_window = (-0.2, 2.0)
baseline = (None, 0)  # use pre-stimulus period for baseline correction; set to None to skip baseline correction
evoked_by = ["animal"]

epochs_apice, evokeds_apice, summary_segmentation, report_segmentation = segment_default_pipeline(
    raw_apice_single, 
    kwargs_events_from_annotations_for_segmentation, 
    event_time_window,
    file_name="test_recording-raw-preproc",
    l_freq=l_freq_epochs,
    h_freq=h_freq_epochs,
    baseline=baseline, 
    kwargs_events_from_annotations_for_metadata=None,
    kwargs_make_metadata=None,                             
    evoked_by=evoked_by,
    output_dir=output_dir_segmented,
    save_log=False,
    save_epochs=True,
    save_only_good_epochs=False,
    save_evoked=True,
    save_report=True,
    save_summary=True,
    save_cfg=True,
    set_reference={'ref_channels':'average'},
    cfg_define_bcbt_epochs=cfg_define_bcbt_epochs,
    cfg_spline_channels=cfg_spline_channels,  
    cfg_bad_epochs=cfg_bad_epochs,              
    n_jobs=-1,
    )


print("run_segmentation completed")

## 7) Detailed Single-File Workflow with RawAPICE and EpochsAPICE

This section details wrapper methods and export/reload flow.

Important:
- After `correct_target_pca`, apply a high-pass filter.
- After `correct_spline_segments`, apply a high-pass filter.

This mirrors the order used inside `preprocess_apice_default`.


### Aplly APICE preprocessing steps using default configurations parameters

In [ ]:
# Load raw data
raw_source_file = input_dir / "test_recording-raw.fif"
raw_base = mne.io.read_raw(raw_source_file, preload=False, verbose=False)

# Apply initial preprocessing steps (e.g., drop channels, crop, resample, set montage) to the raw data
raw_base, _ = preprocess_initial_steps(raw_base, 
                                    drop_electrodes=drop_electrodes,
                                    picks=picks,
                                    crop_times=crop_times,
                                    crop_from_beginnning=crop_from_beginnning,
                                    crop_from_end=crop_from_end,
                                    resample_freq=resample_freq,
                                    stim_channels_to_annotations=stim_channels_to_annotations,
                                    montage=montage,
                                    )

# Band pass filter the data to remove drifts and high-frequency noise before artifact detection and correction steps
Filter(raw_base, l_freq=l_freq, h_freq=h_freq, n_jobs=-1)

# Initialize RawAPICE from the preprocessed raw data
raw_demo = RawAPICE(raw_base)

# Apply artifact detection algorithm targeting bad channels
raw_demo.detect_bad_channels()

# Apply artifact detection algorithm targeting glitches
raw_demo.detect_glitches()

# Correct glitches using target PCA-based correction
raw_demo.correct_target_pca()
Filter(raw_demo, l_freq=l_freq, h_freq=None, n_jobs=-1)

# Apply artifact detection algorithm targeting general artifacts (e.g., motion artifacts)
raw_demo.detect_artifacts()

# Correct artifacts using spline interpolation of segments
raw_demo.correct_spline_segments()
Filter(raw_demo, l_freq=l_freq, h_freq=None, n_jobs=-1)

# Interpolate bad channels using spline interpolation across channels
raw_demo.correct_spline_channels()

# Apply artifact detection algorithm targeting general artifacts again after corrections
raw_demo.detect_artifacts()

# Visualize the artifact structure of the raw data after all detections and corrections
fig = raw_demo.plot_artifact_structure(artifact='all')

# Convert rejected times to annotations and plot, then remove rejection related annotations to keep only the original ones
raw_demo.annotate_bads(times=True, channels=False, data=False, corrected=False)
raw_demo.plot(n_channels=len(raw_demo.info['ch_names']), duration=100)
raw_demo.remove_artifacts_annotations()

# Export and reload raw
raw_export_fullpath = output_dir_preproc /"tutorial_raw-preproc-raw.fif"
raw_demo.export(raw_export_fullpath, overwrite=True)
raw_reloaded = load_rawapice(raw_export_fullpath)
print("Reloaded raw type:", type(raw_reloaded))


### Apply APICE segmentation steps using default configurations

In [ ]:
# Segment continuous data, run epoch-level correction and rejection, then export/reload epochs
events, event_id = mne.events_from_annotations(raw_demo, regexp='animal*')
if len(events) > 0:
    epochs_demo = raw_demo.segment_continuous_data(
        events=events,
        event_id=event_id,
        epoching_kwargs=dict(tmin=-0.2, tmax=2.0, baseline=(None, 0), preload=True, reject_by_annotation=False),
    )
    print("is EpochsAPICE:", isinstance(epochs_demo, EpochsAPICE))

    # Update artifact parameters for the segmented data (optional, but recommended to ensure appropriate parameters for epoch-level artifact handling, otherwise defaults are used)
    if cfg_define_bcbt_epochs is not None:
        epochs_demo.update_artifacts_params(**cfg_define_bcbt_epochs)
    
    # Define BadTimes and BadChannels for the segmented data
    epochs_demo.define_bcbt()

    # interpolate bad channels per epochs using spline interpolation
    epochs_demo.correct_spline_channels()

    # define bad epochs based on the defined bad times and channels, as well as distance and GFP criteria, then remove them from the dataset
    epochs_demo.define_bad_epochs(bad_data=1, bad_time=0, bad_channel=0.3, lim_dist=2, lim_gfp=2)
    epochs_demo.remove_bad_epochs()

    # Compute the evoked responses and plot them
    evokeds = epochs_demo.average()
    evokeds.plot()

    # Save the epochs
    epochs_full_name = output_dir_segmented /"tutorial_epochs-epo.fif"
    epochs_demo.export(epochs_full_name, overwrite=True)

    # Reload the epochs to verify export worked correctly
    epochs_reloaded = load_epochapice(epochs_full_name)
    print("Reloaded epochs type:", type(epochs_reloaded))


## 8) Use custom configurations

Some examples are provided on how to run preprocessing steps with custom parameters

#### Create a custom configuration to detect artifacts and run the detection algorithms

In [ ]:
# Initialize a new ArtifactsConfiguration object to define the artifact detection algorithms and parameters for the segmentation step
cfg_obj = ArtifactsConfiguration()

# Add a group of algorithms. 
# It will run up to 3 loops of artifact detection and rejection, with a minimum rejection rate of 0.1% to continue looping. 
cfg_obj.add_algorithm_group('artifacts_amplitude', max_loops=3, min_rejection=0.1, define_bcbt=True)

# Add an algorithm to the group: Amplitude-based artifact detection with specified parameters
cfg_obj.add_algorithm('artifacts_amplitude', 'Amplitude', {
    'bad_data': None,
    'do_reference_data': False,
    'do_zscore': False,
    'thresh_type': 'outliers_per_channel',
    'thresh': [-2, 2],
    'mask': 0,
    'remove_bct': True,
    'remove_bt': True,
    'remove_bc': True,
}, algorithm_name='ArtifactsAmplitude')

# Add another group of algorithms
# It will run up to 3 loops of artifact detection and rejection, with a minimum rejection rate of 0.1% to continue looping.
cfg_obj.add_algorithm_group('artifacts_maxchange500', max_loops=3, min_rejection=0.1, define_bcbt=True)

# Add an algorithm to the group: MaxChange-based artifact detection with specified parameters
cfg_obj.add_algorithm('artifacts_maxchange500', 'MaxChange', {
    'bad_data': None,
    'do_reference_data': False,
    'do_zscore': False,
    'thresh_type': 'outliers_per_channel',
    'thresh': [None, 2.0],
    'time_window': 0.500,
    'time_window_step': 0.100,
    'mask': 0,
    'remove_bct': True,
    'remove_bt': True,
    'remove_bc': True,
}, algorithm_name='MaxChange500ms')

# Add another group of algorithms to modify the rejection matrix.
cfg_obj.add_algorithm_group('artifacts_modify', max_loops=1, min_rejection=0, define_bcbt=True)

# Add algorithms to the group: Short segment-based modifications with specified parameters
cfg_obj.add_algorithm('artifacts_modify', 'ShortBadSegments', {
    'time_limit': 0.050,
}, algorithm_name='ShortBadSegments', post_detection=True)
cfg_obj.add_algorithm('artifacts_modify', 'ShortGoodSegments', {
    'time_limit': 0.500,
}, algorithm_name='ShortGoodSegments', post_detection=True)

# Check the configuration
cfg_obj.check_configuration()

# save the configuration to a JSON file
output_dir_cfg = repo_root / "tests" / "cfgs"
output_dir_cfg.mkdir(parents=True, exist_ok=True)
custom_cfg_path = output_dir_cfg / "tutorial_custom_algorithms.json"
cfg_obj.save_to_json(custom_cfg_path)



In [ ]:
# Run it passing the configuration dictionry defined above.
raw_cfg = raw_demo.copy()
run_algorithms(raw_cfg, cfg_obj.cfg)

# Run providing the JSON file path instead of a dictionary with the configuration
raw_cfg = raw_demo.copy()
run_algorithms(raw_cfg, custom_cfg_path)

#### Run correction using spline interpolation with custom parameters

In [ ]:
# Define a custom configuration for the spline segments correction algorithm, which will be used in the segmentation step. 
spline_segments_cgf = {}
spline_segments_cgf['p'] = 0.5
spline_segments_cgf['p_neighbors'] = 1
spline_segments_cgf['min_good_time'] = 1.00 
spline_segments_cgf['min_intertime'] = 0.050
spline_segments_cgf['mask_time'] = 0.100
spline_segments_cgf['min_segment_time'] = 0.250
spline_segments_cgf['splice_method'] = 1
spline_segments_cgf['parallelize_mode'] = 'auto'
spline_segments_cgf['save_corrected'] = True

# Save to JSON file
custom_cfg_path = output_dir_cfg / "custom_spline_segments_cfg.json"
with open(custom_cfg_path, 'w') as f:
    json.dump(spline_segments_cgf, f, indent=4)

# Run it passing the configuration dictionry defined above.
raw_cfg.correct_spline_segments(spline_segments_cgf)

# Run providing the JSON file path for the spline segments configuration
raw_cfg.correct_spline_segments(custom_cfg_path)



## 9) Configuration Logic: Building Configurations with `standard_conf`

APICE configurations are built with `ArtifactsConfiguration`, which organises the detection pipeline into **groups** of algorithms. Each group is run independently and can iterate in a loop until either the rejection rate drops below a threshold or the maximum number of loops is reached.

### Key concepts

| Concept | What it controls |
|---|---|
| **Group** (`add_algorithm_group`) | A named stage of the pipeline (e.g. `bad_channels_basic`). Each group loops independently. |
| `max_loops` | Maximum number of re-detection iterations for the group. |
| `min_rejection` | Minimum new-rejection percentage required to keep looping. Loop stops early once new rejection falls below this value. |
| `define_bcbt` | Whether to re-derive BadChannels and BadTimes from the BCT matrix after the group finishes. |
| **Algorithm** (`add_algorithm`) | A detection or post-detection step inside a group. |
| `post_detection=True` | Flags a step that modifies the rejection matrix (e.g. expand/shrink bad segments) *after* the main detection loop. |

The `standard_conf` module provides ready-made functions that encapsulate this logic and expose only the high-level parameters. The cells below unpack two of those functions explicitly.


### 9.1) `cfg_detect_bad_channels` — detecting flat and uncorrelated channels

`cfg_detect_bad_channels` builds a **single-loop** group that marks channels as bad based on two complementary criteria:

1. **`FlatChannel`** — detects channels with negligible variance (flat signal) within sliding time windows.
2. **`ChannelCorr`** — detects channels whose correlation with their neighbours is persistently low.

After the detection loop, two **post-detection** steps clean the rejection matrix:
- **`ShortGoodSegments`** — removes isolated short bursts of "good" signal that are surrounded by bad data (to avoid splitting bad segments).
- **`ShortBadSegments`** — removes isolated very short bad segments (likely false positives).

Because channel-level bad marking rarely needs iteration (`max_loops=1`) and we never want to stop early, `min_rejection=0`.


In [ ]:
# ── cfg_detect_bad_channels — explicit inline implementation ────────────────
#
# This replicates exactly what apice.standard_conf.cfg_detect_bad_channels()
# does, with comments explaining every design choice.

from apice.artifacts_rejection import ArtifactsConfiguration

artcfg_bad_channels = ArtifactsConfiguration()

# ── Group: bad_channels_basic ────────────────────────────────────────────────
# max_loops=1   → run exactly once (channel badness is stable across loops)
# min_rejection=0 → never stop early
# define_bcbt=True → recompute BC/BT from the updated BCT matrix after the group
artcfg_bad_channels.add_algorithm_group(
    'bad_channels_basic',
    max_loops=1,
    min_rejection=0,
    define_bcbt=True,
)

# FlatChannel: flags channels that show almost no signal variation within
# sliding 10-second windows (step 5 s). thresh=5 means a channel is bad if
# it is flagged in at least 5 consecutive windows.
artcfg_bad_channels.add_algorithm('bad_channels_basic', 'FlatChannel', {
    'bad_data': None,          # ignore already-rejected data? None = no
    'do_reference_data': False,
    'do_zscore': False,
    'time_window': 10,         # window length (s)
    'time_window_step': 5,     # step between windows (s)
    'min_change': 1e-7,        # minimum amplitude change to be "active" (V)
    'thresh': 5,               # IQR threshold
    'mask': 0,                 # no masking around detections
})

# ChannelCorr: flags channels whose correlation with their top-5 neighbours
# is below 0.4 across 10-second sliding windows (step 5 s).
artcfg_bad_channels.add_algorithm('bad_channels_basic', 'ChannelCorr', {
    'bad_data': None,
    'do_reference_data': False,
    'do_zscore': False,
    'time_window': 10,
    'time_window_step': 5,
    'top_channel_corr': 5,     # number of nearest neighbours used
    'thresh': 0.4,             # minimum acceptable correlation
    'mask': 0,
})

# ── Post-detection steps ─────────────────────────────────────────────────────
# ShortGoodSegments: collapses good windows shorter than 5 s into bad
# (avoids tiny isolated islands of "good" within a bad channel period).
artcfg_bad_channels.add_algorithm('bad_channels_basic', 'ShortGoodSegments', {
    'time_limit': 5,           # merge good segments shorter than this (s)
}, post_detection=True)

# ShortBadSegments: removes bad windows shorter than 0.5 s
# (likely spurious single-window detections).
artcfg_bad_channels.add_algorithm('bad_channels_basic', 'ShortBadSegments', {
    'time_limit': 0.500,       # remove bad segments shorter than this (s)
}, post_detection=True)

# Validate and inspect
artcfg_bad_channels.check_configuration()
print("bad_channels groups:", list(artcfg_bad_channels.cfg.keys()))
print("algorithms in bad_channels_basic:", list(artcfg_bad_channels.cfg['bad_channels_basic']['algorithms'].keys()))

# ── Shortcut via standard_conf ───────────────────────────────────────────────
# The above is equivalent to a single call:
from apice.standard_conf import cfg_detect_bad_channels
cfg_shortcut = cfg_detect_bad_channels()   # returns artcfg.cfg dict
print("\nstd cfg keys:", list(cfg_shortcut.keys()))


### 9.2) `cfg_detect_artifacts_motion` — iterative multi-feature artifact rejection

`cfg_detect_artifacts_motion` implements the full motion-artifact detection pipeline. It uses **seven groups** with adaptive early-stopping:

#### Early-stopping with `min_rejection_from_thresh`

Rather than setting `min_rejection` manually, the function computes the **expected rejection rate under a Gaussian assumption** given the IQR multiplier (`rejection_level`). The loop stops as soon as the new rejection in a pass drops below this theoretical minimum, i.e. the signal is clean enough that further looping would only reject noise fluctuations.

Two threshold shapes are used:
- **`thresh_sym = [-rejection_level, rejection_level]`** — symmetric, rejects both positive and negative outliers (used for amplitude).
- **`thresh_upper = [None, rejection_level]`** — upper-tail only, rejects only positive outliers (used for MaxChange, which is always positive).

#### Group structure

| Group | Algorithm | Reference | Purpose |
|---|---|---|---|
| `huge_amplitude_abs` | Amplitude (absolute) | — | Remove channels/times exceeding a fixed voltage ceiling (e.g. 1 mV). Single pass, no early-stopping. |
| `artifacts_amplitude` | Amplitude (IQR per channel) | per-channel | Iteratively reject amplitude outliers relative to each channel's own distribution. |
| `artifacts_maxchange500` | MaxChange 500 ms (IQR per channel) | per-channel | Reject segments with large voltage steps over 500 ms windows. |
| `artifacts_maxchange100` | MaxChange 100 ms (IQR per channel) | per-channel | Same over shorter 100 ms windows (catches faster transients). |
| `artifacts_amplitude_avgref` | Amplitude (IQR all) | average reference | Repeat amplitude rejection after re-referencing to the average, to catch spatially diffuse artefacts. |
| `artifacts_maxchange500_avgref` | MaxChange 500 ms (IQR all) | average reference | MaxChange after average re-reference. |
| `artifacts_maxchange100_avgref` | MaxChange 100 ms (IQR all) | average reference | Same over 100 ms. |
| `artifacts_modify` | ShortBad / ShortGood | — | Post-detection clean-up: remove tiny bad or good islands. |


In [ ]:
# ── cfg_detect_artifacts_motion — explicit inline implementation ─────────────
#
# Parameters (matching standard_conf defaults):
rejection_level  = 3        # IQR multiplier for outlier thresholds
min_rejection    = None     # None → compute from Gaussian assumption
max_loops        = 5        # maximum detection-rejection iterations per group
abs_thresh_amp   = 1000e-6  # absolute amplitude ceiling (1 mV in volts)

# ── Threshold shapes ─────────────────────────────────────────────────────────
thresh_sym   = [-rejection_level,  rejection_level]  # both tails (amplitude)
thresh_upper = [None,              rejection_level]  # upper tail only (MaxChange)

# ── Expected minimum rejection under a Gaussian distribution ─────────────────
# If a data sample is truly Gaussian, the fraction rejected by an IQR-based
# threshold is deterministic.  We use this as the early-stopping criterion:
# once a loop rejects less than this amount, the data are effectively clean.
from apice.standard_conf import min_rejection_from_thresh

min_rej_sym   = min_rejection if min_rejection is not None else min_rejection_from_thresh(thresh_sym)
min_rej_upper = min_rejection if min_rejection is not None else min_rejection_from_thresh(thresh_upper)

print(f"thresh_sym   {thresh_sym}   → min_rejection = {min_rej_sym:.5g} %")
print(f"thresh_upper {thresh_upper} → min_rejection = {min_rej_upper:.5g} %")

# ── Build configuration ───────────────────────────────────────────────────────
from apice.artifacts_rejection import ArtifactsConfiguration

artcfg_motion = ArtifactsConfiguration()

# ── Group 1: huge_amplitude_abs — hard ceiling ───────────────────────────────
# Removes data that exceeds an absolute voltage limit (e.g. amplifier saturation).
# Single pass (max_loops=1), never stops early (min_rejection=0).
artcfg_motion.add_algorithm_group('huge_amplitude_abs', max_loops=1, min_rejection=0, define_bcbt=True)
artcfg_motion.add_algorithm('huge_amplitude_abs', 'Amplitude', {
    'bad_data': None,
    'do_reference_data': False,
    'do_zscore': False,
    'thresh_type': 'absolute',
    'thresh': abs_thresh_amp,    # fixed ceiling in volts
    'mask': 0,
    'remove_bct': True,
    'remove_bt': True,
    'remove_bc': True,
}, algorithm_name='HugeAmplitudeAbsolute')

# ── Group 2: artifacts_amplitude — per-channel amplitude outliers ─────────────
# Symmetric IQR threshold: rejects time points where amplitude deviates more
# than rejection_level × IQR below Q1 or above Q3, computed per channel.
artcfg_motion.add_algorithm_group('artifacts_amplitude', max_loops=max_loops, min_rejection=min_rej_sym, define_bcbt=True)
artcfg_motion.add_algorithm('artifacts_amplitude', 'Amplitude', {
    'bad_data': None,
    'do_reference_data': False,
    'do_zscore': False,
    'thresh_type': 'outliers_per_channel',
    'thresh': thresh_sym,
    'mask': 0,
    'remove_bct': True,
    'remove_bt': True,
    'remove_bc': True,
}, algorithm_name='ArtifactsAmplitude')

# ── Group 3: artifacts_maxchange500 — voltage steps over 500 ms ───────────────
# Detects rapid large voltage transitions (e.g. movement bumps) using a
# sliding 500 ms window. Upper-tail only because MaxChange is always ≥ 0.
artcfg_motion.add_algorithm_group('artifacts_maxchange500', max_loops=max_loops, min_rejection=min_rej_upper, define_bcbt=True)
artcfg_motion.add_algorithm('artifacts_maxchange500', 'MaxChange', {
    'bad_data': None,
    'do_reference_data': False,
    'do_zscore': False,
    'thresh_type': 'outliers_per_channel',
    'thresh': thresh_upper,
    'time_window': 0.500,
    'time_window_step': 0.100,
    'mask': 0,
    'remove_bct': True,
    'remove_bt': True,
    'remove_bc': True,
}, algorithm_name='MaxChange500ms')

# ── Group 4: artifacts_maxchange100 — voltage steps over 100 ms ───────────────
# Same as above but with a shorter window to catch faster transients.
artcfg_motion.add_algorithm_group('artifacts_maxchange100', max_loops=max_loops, min_rejection=min_rej_upper, define_bcbt=True)
artcfg_motion.add_algorithm('artifacts_maxchange100', 'MaxChange', {
    'bad_data': None,
    'do_reference_data': False,
    'do_zscore': False,
    'thresh_type': 'outliers_per_channel',
    'thresh': thresh_upper,
    'time_window': 0.100,
    'time_window_step': 0.020,
    'mask': 0,
    'remove_bct': True,
    'remove_bt': True,
    'remove_bc': True,
}, algorithm_name='MaxChange100ms')

# ── Groups 5–7: average-reference versions ────────────────────────────────────
# Repeating groups 2–4 after re-referencing to the average catches spatially
# diffuse artifacts that are attenuated in a single-channel reference.
# 'thresh_type': 'outliers_all' pools all channels/times together for thresholding.

artcfg_motion.add_algorithm_group('artifacts_amplitude_avgref', max_loops=max_loops, min_rejection=min_rej_sym, define_bcbt=True)
artcfg_motion.add_algorithm('artifacts_amplitude_avgref', 'Amplitude', {
    'bad_data': 'replace by nan',  # exclude already-bad data from the reference
    'do_reference_data': True,     # subtract average reference before thresholding
    'do_zscore': False,
    'thresh_type': 'outliers_all',
    'thresh': thresh_sym,
    'mask': 0,
    'remove_bct': True,
    'remove_bt': True,
    'remove_bc': True,
}, algorithm_name='ArtifactsAmplitudeAvgRef')

artcfg_motion.add_algorithm_group('artifacts_maxchange500_avgref', max_loops=max_loops, min_rejection=min_rej_upper, define_bcbt=True)
artcfg_motion.add_algorithm('artifacts_maxchange500_avgref', 'MaxChange', {
    'bad_data': 'replace by nan',
    'do_reference_data': True,
    'do_zscore': False,
    'thresh_type': 'outliers_all',
    'thresh': thresh_upper,
    'time_window': 0.500,
    'time_window_step': 0.100,
    'mask': 0,
    'remove_bct': True,
    'remove_bt': True,
    'remove_bc': True,
}, algorithm_name='MaxChange500msAvgRef')

artcfg_motion.add_algorithm_group('artifacts_maxchange100_avgref', max_loops=max_loops, min_rejection=min_rej_upper, define_bcbt=True)
artcfg_motion.add_algorithm('artifacts_maxchange100_avgref', 'MaxChange', {
    'bad_data': 'replace by nan',
    'do_reference_data': True,
    'do_zscore': False,
    'thresh_type': 'outliers_all',
    'thresh': thresh_upper,
    'time_window': 0.100,
    'time_window_step': 0.020,
    'mask': 0,
    'remove_bct': True,
    'remove_bt': True,
    'remove_bc': True,
}, algorithm_name='MaxChange100msAvgRef')

# ── Group 8: artifacts_modify — post-detection clean-up ──────────────────────
# Four sequential post-detection steps tidy the rejection matrix:
#  1. Remove bad segments < 20 ms  (very short spurious detections)
#  2. Remove good segments < 20 ms (isolated tiny good islands within bad)
#  3. Remove bad segments < 50 ms  (slightly longer spurious detections)
#  4. Remove good segments < 500 ms (short good islands — keep only stable clean stretches)
artcfg_motion.add_algorithm_group('artifacts_modify', max_loops=1, min_rejection=0, define_bcbt=True)
artcfg_motion.add_algorithm('artifacts_modify', 'ShortBadSegments',  {'time_limit': 0.020}, algorithm_name='VeryShortBadSegments',  post_detection=True)
artcfg_motion.add_algorithm('artifacts_modify', 'ShortGoodSegments', {'time_limit': 0.020}, algorithm_name='VeryShortGoodSegments', post_detection=True)
artcfg_motion.add_algorithm('artifacts_modify', 'ShortBadSegments',  {'time_limit': 0.050}, algorithm_name='ShortBadSegments',       post_detection=True)
artcfg_motion.add_algorithm('artifacts_modify', 'ShortGoodSegments', {'time_limit': 0.500}, algorithm_name='ShortGoodSegments',      post_detection=True)

artcfg_motion.check_configuration()
print("Groups:", list(artcfg_motion.cfg.keys()))

# ── Shortcut via standard_conf ───────────────────────────────────────────────
# The above is equivalent to:
from apice.standard_conf import cfg_detect_artifacts_motion
cfg_motion_shortcut = cfg_detect_artifacts_motion(rejection_level=3, max_loops=5)
print("\nGroups via standard_conf:", list(cfg_motion_shortcut.keys()))
